# 02 - Containerization and EKS Partition Planning (Lab Support)

Use this notebook to design a partition-parallel EKS batch strategy and generate a manifest scaffold.

In [ ]:
from pathlib import Path
import pandas as pd

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

monthly_files = sorted(REPO_ROOT.glob('trips_2023_*.csv'))
months = [f.stem.split('_')[-1] for f in monthly_files]

mapping = pd.DataFrame({
    'job_completion_index': list(range(len(months))),
    'month_partition': months,
    'raw_input_prefix': [f's3://<BUCKET>/raw/month={m}/' for m in months],
    'clean_output_prefix': [f's3://<BUCKET>/cleaned/month={m}/cleaned.parquet' for m in months],
})
mapping

In [ ]:
from textwrap import dedent

ACCOUNT_ID = '<ACCOUNT_ID>'
REGION = 'us-east-1'
BUCKET = '<BUCKET>'

manifest = dedent(f'''apiVersion: batch/v1
kind: Job
metadata:
  name: trip-cleaning-batch
spec:
  parallelism: {len(months)}
  completions: {len(months)}
  completionMode: Indexed
  backoffLimit: 2
  template:
    spec:
      restartPolicy: Never
      containers:
        - name: cleaner
          image: {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/trip-cleaner:latest
          command: ["/bin/sh", "-c"]
          args:
            - |
              set -eu
              IDX="${{JOB_COMPLETION_INDEX:-0}}"
              MONTH=$(printf "%02d" $((IDX + 1)))
              python clean_trips.py \
                --input s3://{BUCKET}/raw/month=${{MONTH}}/ \
                --output s3://{BUCKET}/cleaned/month=${{MONTH}}/cleaned.parquet
          env:
            - name: JOB_COMPLETION_INDEX
              valueFrom:
                fieldRef:
                  fieldPath: metadata.annotations['batch.kubernetes.io/job-completion-index']
            - name: AWS_DEFAULT_REGION
              value: {REGION}
''')

print(manifest)

In [ ]:
out_path = REPO_ROOT / 'notebooks' / 'output' / 'cleaning-job.generated.yaml'
out_path.write_text(manifest, encoding='utf-8')
print('Wrote:', out_path)

In [ ]:
checklist = [
    'Build and test container locally',
    'Push tagged image to ECR',
    'Fill in ACCOUNT_ID and BUCKET placeholders',
    'Apply job and monitor pods/logs',
    'Verify one cleaned output per partition',
    'Delete EKS cluster after completion',
]

for i, item in enumerate(checklist, 1):
    print(f'{i}. {item}')

## Reflection Prompt
1. Why is indexed completion useful for partitioned data?
2. What needs to be idempotent for safe retries?
3. Which failure class is most likely in your setup: IAM, image pull, or path mismatch?